# Prompt Engineering Experiments

5 strategies x 3 frontier models.

In [1]:
import pandas as pd, numpy as np, os, warnings
from scipy import stats
warnings.filterwarnings('ignore')

PROMPT_DIR = None
for d in ['../results/prompts','results/prompts']:
    if os.path.isdir(d): PROMPT_DIR = d; break

MODES = ['standard','cot','structured','minimal','few-shot']
ML = {'standard':'Standard','cot':'Chain-of-thought','structured':'Structured checklist',
      'minimal':'Minimal','few-shot':'Few-shot'}

frames = []
for f in sorted(os.listdir(PROMPT_DIR)):
    if not f.endswith('.csv'): continue
    df = pd.read_csv(os.path.join(PROMPT_DIR, f))
    for col in ['valid','correct','expected_valid']:
        if col in df.columns: df[col] = df[col].astype(str).str.strip().str.lower()=='true'
    for mode in MODES:
        if f'-{mode}.csv' in f: df['prompt_mode']=mode; break
    frames.append(df)
data = pd.concat(frames, ignore_index=True)
models = sorted(data['model'].unique())
print(f"Loaded: {len(models)} models x {data['prompt_mode'].nunique()} strategies")

Loaded: 3 models x 5 strategies


## Accuracy by Strategy and Model

In [2]:
rows = []
for model in models:
    for mode in MODES:
        mdf = data[(data['model']==model)&(data['prompt_mode']==mode)]
        if len(mdf)==0: continue
        n=len(mdf); k=int(mdf['correct'].sum()); acc=k/n
        tp=int((mdf['valid']&mdf['expected_valid']).sum())
        fp=int((mdf['valid']&~mdf['expected_valid']).sum())
        tn=int((~mdf['valid']&~mdf['expected_valid']).sum())
        fn=int((~mdf['valid']&mdf['expected_valid']).sum())
        spec=tn/(tn+fp) if (tn+fp) else 0
        rows.append({'Model':model,'Strategy':ML.get(mode,mode),'Accuracy':acc,'Specificity':spec})
pdf = pd.DataFrame(rows)
pivot = pdf.pivot(index='Strategy',columns='Model',values='Accuracy')
pivot = pivot[models].reindex([ML[m] for m in MODES if ML[m] in pivot.index])
pivot.style.format('{:.1%}').background_gradient(cmap='RdYlGn',vmin=0.5,vmax=1.0).set_caption(
    'Prompt Strategy: Accuracy')

Model,gpt-oss-120b,gpt-oss-20b,llama-3.3-70b
Strategy,,,
Standard,87.1%,87.1%,90.3%
Chain-of-thought,51.6%,61.3%,90.3%
Structured checklist,48.4%,54.8%,48.4%
Minimal,48.4%,48.4%,87.1%
Few-shot,90.3%,93.5%,87.1%


## Specificity (Error Detection Rate)

In [3]:
spec_pivot = pdf.pivot(index='Strategy',columns='Model',values='Specificity')
spec_pivot = spec_pivot[models].reindex([ML[m] for m in MODES if ML[m] in spec_pivot.index])
spec_pivot.style.format('{:.2f}').background_gradient(cmap='RdYlGn',vmin=0,vmax=1).set_caption(
    'Prompt Strategy: Specificity (error detection)')

Model,gpt-oss-120b,gpt-oss-20b,llama-3.3-70b
Strategy,,,
Standard,0.88,0.81,0.88
Chain-of-thought,0.06,0.38,0.88
Structured checklist,0.00,0.19,0.06
Minimal,0.00,0.00,0.88
Few-shot,0.88,0.88,0.88


## Recommendation

In [4]:
print("Standard prompt is recommended:")
print("  - Highest or tied-highest accuracy for all models")
print("  - Fastest inference time")
print("  - CoT actually HURTS MoE models (reduces specificity)")
print("  - Structured/minimal degrade to base-rate performance")

Standard prompt is recommended:
  - Highest or tied-highest accuracy for all models
  - Fastest inference time
  - CoT actually HURTS MoE models (reduces specificity)
  - Structured/minimal degrade to base-rate performance
